In [ ]:
# 本文件说明Fama_MacBeth回归的使用
%run config_0.ipynb
%run data_io_1.ipynb
%run clean_and_universe_2.ipynb
%run features_3.ipynb
%run monthly_4.ipynb
%run portfolio_5.ipynb
%run time_series_regression_6.ipynb

import pandas as pd
import numpy as np
import statsmodels.api as sm

In [ ]:
# Fama-MacBeth 第二步：每一期做一次截面回归。
#
# 同样要用 B.T 而不是 B：
#   第一步时序回归 R(T×N) = F(T×K) @ B(K×N) 里，B 是 K×N，
#   每一列是某只股票的 K 个因子暴露；
#   第二步截面回归里，被解释变量是某一期的 N 只股票收益 R_t (1×N)，
#   解释变量是这 N 只股票各自的因子暴露，设计矩阵必须 N 行 × K 列，
#   也就是 B 的转置 B.T（N×K）。
def fama_macbeth(R , B):
    # 设计矩阵：N x (K+1)。B 对所有期都一样，所以放在循环外只算一次
    X = sm.add_constant(B.T).reindex(R.columns)

    F_t_ls = []
    alpha_ls = []
    for t in range(R.shape[0]):
        R_t = R.iloc[t]                          # 第 t 期截面收益（N 只股票）
        valid = R_t.notna() & X.notna().all(axis = 1)
        res = sm.OLS(R_t[valid] , X[valid]).fit()
        F_t_ls.append(res.params.drop('const'))  # 该期因子溢价 λ_t (K,)
        alpha_ls.append(res.params['const'])     # 该期截距

    F = pd.DataFrame(F_t_ls , index = R.index)      # T x K：每期一个因子溢价
    Alpha = pd.Series(alpha_ls , index = R.index)   # T：每期截距

    # 对 λ_t 序列做 Newey-West 调整：因子溢价 λ̄ 与 t 值（逐因子算，避免多元 OLS 的 HAC 问题）
    lambda_bar = pd.Series(index = F.columns , dtype = float)
    t_stat = pd.Series(index = F.columns , dtype = float)
    ones = np.ones((F.shape[0] , 1))
    for k in F.columns:
        res = sm.OLS(F[k].to_numpy() , ones).fit(cov_type = 'HAC' , cov_kwds = {'maxlags': 4})
        lambda_bar[k] = res.params[0]             # λ̄ (K,)
        t_stat[k] = res.tvalues[0]                # Newey-West t 值 (K,)
    return F , Alpha , lambda_bar , t_stat


df = read_daily_data()
df = cleaning(df)
df = universe(df)
cf_df = read_cf_data()
cf_df = cf_cleaning(cf_df)
bs_df = read_bs_data()
bs_df = bs_cleaning(bs_df)
df = get_market_feature(df)
df = get_momentum(df)   # 与 monthly_4 / portfolio_5 的内联流程一致：动量列必须先构造
df = get_cp(df , cf_df)
df = get_asset_growth(df , bs_df)
df = get_excess_return(df)
df = get_bm(df)
df = get_ep(df)
df = get_one_month_volatility(df)
df = get_max_return(df)
df = get_illiquidity(df)
df = get_turnover_12m(df)
df = get_abn_turnover_1m(df)
monthly = get_ym_monthly_excessret(df)
snap = get_form_signal(df)
monthly_final = get_monthly_panel(monthly , snap)
F = get_groups_neutral(monthly_final)
R , T , N= get_R(monthly)
B , _ , _ = time_series_regression(R , F)
F , Alpha , lambda_bar , t_stat = fama_macbeth(R , B)
print('查看 Fama-MacBeth 每期因子溢价 λ_t')
print(F)
print('查看因子溢价 λ̄ 与 Newey-West t 值')
print(lambda_bar)
print(t_stat)